In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('../data/raw/ethiopia_fi_unified_data - ethiopia_fi_unified_data.csv')
ref = pd.read_csv('../data/raw/reference_codes - reference_codes.csv')

df.head()
ref.head()

,field,code,description,applies_to
0,record_type,observation,Actual measured value from a source,All
1,record_type,event,Policy launch market event or milestone,All
2,record_type,impact_link,Relationship between event and indicator (link...,All
3,record_type,target,Policy target or official goal,All
4,record_type,baseline,Starting point for comparison,All


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   record_id            43 non-null     object 
 1   record_type          43 non-null     object 
 2   category             10 non-null     object 
 3   pillar               33 non-null     object 
 4   indicator            43 non-null     object 
 5   indicator_code       43 non-null     object 
 6   indicator_direction  33 non-null     object 
 7   value_numeric        33 non-null     float64
 8   value_text           10 non-null     object 
 9   value_type           43 non-null     object 
 10  unit                 33 non-null     object 
 11  observation_date     43 non-null     object 
 12  period_start         10 non-null     object 
 13  period_end           10 non-null     object 
 14  fiscal_year          43 non-null     object 
 15  gender               43 non-null     objec

In [5]:
df["record_type"].value_counts()

record_type
observation    30
event          10
target          3
Name: count, dtype: int64

In [6]:
df["pillar"].value_counts(dropna=False)

pillar
ACCESS           16
USAGE            11
NaN              10
GENDER            5
AFFORDABILITY     1
Name: count, dtype: int64

In [7]:
df["source_type"].value_counts(dropna=False)

source_type
operator      15
survey        10
regulator      7
research       4
policy         3
calculated     2
news           2
Name: count, dtype: int64

In [8]:
df["confidence"].value_counts(dropna=False)

confidence
high      40
medium     3
Name: count, dtype: int64

In [9]:
df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")

df["observation_date"].min(), df["observation_date"].max()


(Timestamp('2014-12-31 00:00:00'), Timestamp('2030-12-31 00:00:00'))

In [10]:
coverage = (
    df[df["record_type"] == "observation"]
    .groupby("indicator_code")
    .agg(
        first_year=("observation_date", lambda x: x.min().year),
        last_year=("observation_date", lambda x: x.max().year),
        count=("value_numeric", "count")
    )
    .reset_index()
)

coverage.sort_values("count")


,indicator_code,first_year,last_year,count
3,ACC_MOBILE_PEN,2025,2025,1
5,AFF_DATA_INCOME,2024,2024,1
7,GEN_GAP_MOBILE,2024,2024,1
12,USG_CROSSOVER,2025,2025,1
11,USG_ATM_VALUE,2025,2025,1
10,USG_ATM_COUNT,2025,2025,1
9,USG_ACTIVE_RATE,2024,2024,1
8,GEN_MM_SHARE,2024,2024,1
14,USG_MPESA_USERS,2024,2024,1
13,USG_MPESA_ACTIVE,2024,2024,1


In [11]:
df[df["record_type"] == "observation"][
    ["indicator_code", "indicator", "pillar"]
].drop_duplicates().sort_values(["pillar", "indicator_code"])


,indicator_code,indicator,pillar
8,ACC_4G_COV,4G Population Coverage,ACCESS
11,ACC_FAYDA,Fayda Digital ID Enrollment,ACCESS
6,ACC_MM_ACCOUNT,Mobile Money Account Rate,ACCESS
10,ACC_MOBILE_PEN,Mobile Subscription Penetration,ACCESS
0,ACC_OWNERSHIP,Account Ownership Rate,ACCESS
25,AFF_DATA_INCOME,Data Affordability Index,AFFORDABILITY
26,GEN_GAP_ACC,Account Ownership Gender Gap,GENDER
29,GEN_GAP_MOBILE,Mobile Phone Gender Gap,GENDER
28,GEN_MM_SHARE,Female Mobile Money Account Share,GENDER
24,USG_ACTIVE_RATE,Mobile Money Activity Rate,USAGE


In [13]:
date_col = next(
    col for col in ["event_date", "observation_date", "date"]
    if col in df.columns
)

events_sorted = (
    df[df["record_type"] == "event"]
    .sort_values(date_col)
)

events_sorted


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
33,EVT_0001,event,product_launch,NaN,Telebirr Launch,EVT_TELEBIRR,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,First major mobile money service in Ethiopia,NaN
41,EVT_0009,event,policy,NaN,NFIS-II Strategy Launch,EVT_NFIS2,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,5-year national financial inclusion strategy,NaN
34,EVT_0002,event,market_entry,NaN,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,End of state telecom monopoly,NaN
35,EVT_0003,event,product_launch,NaN,M-Pesa Ethiopia Launch,EVT_MPESA,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Second mobile money entrant,NaN
36,EVT_0004,event,infrastructure,NaN,Fayda Digital ID Program Rollout,EVT_FAYDA,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,National biometric digital ID system,NaN
37,EVT_0005,event,policy,NaN,Foreign Exchange Liberalization,EVT_FX_REFORM,NaN,NaN,Implemented,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Birr float introduced,NaN
38,EVT_0006,event,milestone,NaN,P2P Transaction Count Surpasses ATM,EVT_CROSSOVER,NaN,NaN,Achieved,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Historic: digital > cash for first time,NaN
39,EVT_0007,event,partnership,NaN,M-Pesa EthSwitch Integration,EVT_MPESA_INTEROP,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Full interoperability for M-Pesa,NaN
42,EVT_0010,event,pricing,NaN,Safaricom Ethiopia Price Increase,EVT_SAFCOM_PRICE,NaN,NaN,Implemented,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Data and voice prices increased 20-82%,NaN
40,EVT_0008,event,infrastructure,NaN,EthioPay Instant Payment System Launch,EVT_ETHIOPAY,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,National real-time payment system,NaN


In [14]:
events = df[df["record_type"] == "event"]
events.sort_values("observation_date")

,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
33,EVT_0001,event,product_launch,NaN,Telebirr Launch,EVT_TELEBIRR,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,First major mobile money service in Ethiopia,NaN
41,EVT_0009,event,policy,NaN,NFIS-II Strategy Launch,EVT_NFIS2,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,5-year national financial inclusion strategy,NaN
34,EVT_0002,event,market_entry,NaN,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,End of state telecom monopoly,NaN
35,EVT_0003,event,product_launch,NaN,M-Pesa Ethiopia Launch,EVT_MPESA,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Second mobile money entrant,NaN
36,EVT_0004,event,infrastructure,NaN,Fayda Digital ID Program Rollout,EVT_FAYDA,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,National biometric digital ID system,NaN
37,EVT_0005,event,policy,NaN,Foreign Exchange Liberalization,EVT_FX_REFORM,NaN,NaN,Implemented,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Birr float introduced,NaN
38,EVT_0006,event,milestone,NaN,P2P Transaction Count Surpasses ATM,EVT_CROSSOVER,NaN,NaN,Achieved,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Historic: digital > cash for first time,NaN
39,EVT_0007,event,partnership,NaN,M-Pesa EthSwitch Integration,EVT_MPESA_INTEROP,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Full interoperability for M-Pesa,NaN
42,EVT_0010,event,pricing,NaN,Safaricom Ethiopia Price Increase,EVT_SAFCOM_PRICE,NaN,NaN,Implemented,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Data and voice prices increased 20-82%,NaN
40,EVT_0008,event,infrastructure,NaN,EthioPay Instant Payment System Launch,EVT_ETHIOPAY,NaN,NaN,Launched,categorical,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,National real-time payment system,NaN


In [17]:
new_observations = pd.DataFrame([
    {
        "record_type": "observation",
        "pillar": "usage",
        "indicator": "Smartphone penetration (% adults)",
        "indicator_code": "ENB_SMARTPHONE_PCT",
        "value_numeric": 38,
        "observation_date": "2023-01-01",
        "source_name": "ITU",
        "source_type": "international_org",
        "source_url": "https://www.itu.int",
        "confidence": "medium",
        "original_text": "Estimated smartphone penetration in Ethiopia ~38% in 2023",
        "collected_by": "Ratha",
        "collection_date": "2026-02-03",
        "notes": "Key enabler for digital payment usage"
    },
    {
        "record_type": "observation",
        "pillar": "usage",
        "indicator": "4G population coverage (%)",
        "indicator_code": "ENB_4G_COVERAGE_PCT",
        "value_numeric": 66,
        "observation_date": "2023-01-01",
        "source_name": "GSMA",
        "source_type": "industry_report",
        "source_url": "https://www.gsma.com",
        "confidence": "medium",
        "original_text": "4G population coverage estimated at ~66%",
        "collected_by": "Ratha",
        "collection_date": "2026-02-03",
        "notes": "Infrastructure enabler for mobile payments"
    }
])

df = pd.concat([df, new_observations], ignore_index=True)


In [18]:
new_events = pd.DataFrame([
    {
        "record_type": "event",
        "id": "EVENT_FAYDA_2023",
        "event_name": "Fayda Digital ID National Rollout",
        "category": "infrastructure",
        "event_date": "2023-06-01",
        "description": "National rollout of Fayda digital identification system",
        "source_name": "Government of Ethiopia",
        "source_type": "government",
        "source_url": "https://id.gov.et",
        "confidence": "high",
        "original_text": "Fayda digital ID launched nationally in 2023",
        "collected_by": "Ratha",
        "collection_date": "2026-02-03",
        "notes": "Expected to reduce KYC barriers to account ownership"
    }
])

df = pd.concat([df, new_events], ignore_index=True)


In [19]:
new_impact_links = pd.DataFrame([
    {
        "record_type": "impact_link",
        "parent_id": "EVENT_FAYDA_2023",
        "pillar": "access",
        "related_indicator": "ACC_OWNERSHIP",
        "impact_direction": "positive",
        "impact_magnitude": 0.02,
        "lag_months": 18,
        "evidence_basis": "Digital ID rollouts in India and Rwanda"
    }
])

df = pd.concat([df, new_impact_links], ignore_index=True)


In [20]:
df["record_type"].value_counts()

record_type
observation    32
event          11
target          3
impact_link     1
Name: count, dtype: int64

In [23]:
df.to_csv('../data/processed/processed_data.csv', index=False)